# 03 · Ablation Analysis — Classical DIP on Sen1Floods11

This notebook runs the 32-config ablation on the Sen1Floods11 test split, inspects the results, performs significance tests between the top methods, and renders the 4×4 qualitative comparison grid required by the rubric (§4 Analysis & Results, 20 %).

Outputs written to:
- `results/ablation.csv` — table of 32 rows
- `results/per_chip/*.npz` — per-chip IoU arrays
- `reports/figures/phase3_*.png` — visualisations
- `reports/phase3_report.md` is updated separately with the winning config.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd() if (Path.cwd() / 'PRD.md').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.sen1floods11_loader import Sen1Floods11Dataset
from src.eval.ablation import run_ablation, predict, all_configs, load_per_chip_iou, AblationConfig
from src.eval.significance import mcnemar_test, paired_bootstrap_iou

FIG = Path('reports/figures'); FIG.mkdir(parents=True, exist_ok=True)
RESULTS = Path('results'); RESULTS.mkdir(parents=True, exist_ok=True)

SEN1FLOODS11_ROOT = Path(os.environ.get(
    'SEN1FLOODS11_DIR',
    '/content/drive/MyDrive/dda/sen1floods11'
))
assert SEN1FLOODS11_ROOT.exists(), f'Sen1Floods11 not found at {SEN1FLOODS11_ROOT}'

ds = Sen1Floods11Dataset(SEN1FLOODS11_ROOT, split='test', modality='s2')
print('test split size:', len(ds))

## 1 · Run the 32-config ablation

On a free-tier Colab CPU this takes roughly 3–5 minutes. Result CSV is cached; re-running only overwrites it.

In [ ]:
df = run_ablation(ds, results_dir=RESULTS)
df.sort_values('mean_iou', ascending=False).head(10)

## 2 · Visualise the result table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
sorted_df = df.sort_values('mean_iou', ascending=True)
y = np.arange(len(sorted_df))
ax.barh(y, sorted_df['mean_iou'],
        color=['#4c72b0' if m else '#c44e52' for m in sorted_df['morphology']])
ax.set_yticks(y); ax.set_yticklabels(sorted_df['config'], fontsize=8)
ax.set_xlabel('mean IoU on Sen1Floods11 test')
ax.set_title('Classical DIP ablation — sorted by IoU')
ax.grid(axis='x', alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#4c72b0', label='+ morphology'),
    Patch(color='#c44e52', label='raw threshold'),
], loc='lower right')
plt.tight_layout(); plt.savefig(FIG / 'phase3_ablation_bars.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Heatmap over the index × threshold grid, averaged over morph on/off.
pivot = df.groupby(['index', 'threshold'])['mean_iou'].mean().unstack()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i, j]:.3f}', ha='center', va='center', color='black', fontsize=9)
plt.colorbar(im, ax=ax, label='mean IoU')
ax.set_title('Mean IoU · index (row) × threshold (col), avg over morph')
plt.tight_layout(); plt.savefig(FIG / 'phase3_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

## 3 · Pick the winning config + McNemar vs runner-up

In [ ]:
top2 = df.sort_values('mean_iou', ascending=False).head(2)
winner_name = top2.iloc[0]['config']
runner_name = top2.iloc[1]['config']
print('Winner:     ', winner_name, f"IoU={top2.iloc[0]['mean_iou']:.4f}")
print('Runner-up:  ', runner_name, f"IoU={top2.iloc[1]['mean_iou']:.4f}")

# Load per-chip IoU for bootstrap CI
iou_winner = load_per_chip_iou(winner_name)
iou_runner = load_per_chip_iou(runner_name)

boot = paired_bootstrap_iou(iou_winner, iou_runner, n_bootstrap=10_000)
print(f"\nΔIoU mean = {boot.mean_delta:+.4f}")
print(f"  {int(boot.confidence*100)}% CI = [{boot.ci_lower:+.4f}, {boot.ci_upper:+.4f}]")

In [ ]:
# McNemar's test needs predictions at pixel level, so rebuild masks for the
# top-2 configs on a small subsample.
import re
def parse_config_name(name: str) -> AblationConfig:
    m = re.match(r'^(?P<idx>[a-z_]+?)_(?P<thr>otsu|triangle|yen|li)_(?P<morph>morph|raw)$', name)
    assert m, name
    return AblationConfig(index=m['idx'], threshold=m['thr'], morphology=m['morph'] == 'morph')

cfg_a = parse_config_name(winner_name)
cfg_b = parse_config_name(runner_name)

preds_a, preds_b, labels = [], [], []
for i in range(min(30, len(ds))):
    s = ds[i]
    img, y = s['image'].numpy(), s['label'].numpy()
    preds_a.append(predict(img, cfg_a))
    preds_b.append(predict(img, cfg_b))
    labels.append(y)

pa = np.concatenate([p.ravel() for p in preds_a])
pb = np.concatenate([p.ravel() for p in preds_b])
yy = np.concatenate([y.ravel() for y in labels])
mc = mcnemar_test(pa, pb, yy)
print(f"McNemar χ² = {mc.statistic:.2f}, p = {mc.p_value:.4g} → " +
      ('SIGNIFICANT at α=0.05' if mc.significant() else 'not significant'))

## 4 · Qualitative 4×4 comparison grid

Four randomly-chosen chips × four methods (winner, runner-up, MNDWI-Otsu raw, GT) — the figure that anchors the Analysis & Results section of the IEEE report.

In [ ]:
def stretch(x, lo=2, hi=98):
    a, b = np.percentile(x[np.isfinite(x)], [lo, hi])
    return np.clip((x - a) / (b - a + 1e-9), 0, 1)

rng = np.random.default_rng(0)
chip_ids = rng.choice(len(ds), size=4, replace=False)

cfg_mndwi_otsu = AblationConfig('mndwi', 'otsu', True)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
col_titles = ['RGB', f'Winner\n{winner_name}', f'Runner-up\n{runner_name}', 'Ground truth']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=10)

for row, ci in enumerate(chip_ids):
    s = ds[int(ci)]
    img, y = s['image'].numpy(), s['label'].numpy()
    rgb = np.transpose(np.stack([stretch(img[i]) for i in (2, 1, 0)]), (1, 2, 0))
    mask_win = predict(img, cfg_a)
    mask_run = predict(img, cfg_b)

    axes[row, 0].imshow(rgb); axes[row, 0].set_ylabel(s['chip_id'], fontsize=7)
    axes[row, 1].imshow(rgb); axes[row, 1].imshow(mask_win, cmap='Blues', alpha=0.5)
    axes[row, 2].imshow(rgb); axes[row, 2].imshow(mask_run, cmap='Blues', alpha=0.5)
    y_disp = np.where(y == -1, 0, y)
    axes[row, 3].imshow(rgb); axes[row, 3].imshow(y_disp, cmap='Blues', alpha=0.5)
    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(); plt.savefig(FIG / 'phase3_qualitative_grid.png', dpi=150, bbox_inches='tight'); plt.show()

## 5 · Phase 3 exit criteria

- [ ] `results/ablation.csv` written with 32 rows (or whatever subset of `configs` was chosen).
- [ ] Bar chart + heatmap saved to `reports/figures/phase3_*.png`.
- [ ] Top 2 methods compared via bootstrap CI and McNemar test; result recorded.
- [ ] 4×4 qualitative grid saved.
- [ ] `reports/phase3_report.md` updated with winner and narrative.

Proceed → **Phase 4 · U-Net training**.